# Oferta: autocorrelación y estacionalidad

Se separa la estacionalidad mensual de la oferta de su autocorrelación entre días, para verificar si el ρ = 0.48 del Informe 1 (`RHO_OFERTA` en `codigo/configuracion.py`) corresponde al AR(1) que usa el modelo sobre la oferta estandarizada por mes (`METODOLOGIA.md` §3).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codigo").is_dir())
sys.path.insert(0, str(RAIZ))

from codigo import cargador_datos, distribuciones
from codigo import configuracion as cfg

REZAGOS = [1, 2, 3, 7, 14, 30]
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
DIAS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]

oferta = cargador_datos.cargar_oferta_historica()
oferta["mes"] = oferta["fecha"].dt.month
oferta["anio"] = oferta["fecha"].dt.year
oferta["dia_semana"] = oferta["fecha"].dt.dayofweek

print(f"{len(oferta)} días, del {oferta['fecha'].min().date()} al {oferta['fecha'].max().date()}")

## 1. Autocorrelación de la serie cruda

Se calcula la autocorrelación de la oferta sin transformar y se compara con la que tendría un AR(1) con ρ = 0.48, que decae como $0.48^k$.

In [ ]:
acf_cruda = acf(oferta["oferta"], nlags=30, fft=False)
pd.DataFrame({
    "rezago (días)": REZAGOS,
    "serie cruda": [acf_cruda[k] for k in REZAGOS],
    "AR(1) con ρ = 0.48": [acf_cruda[1] ** k for k in REZAGOS],
}).set_index("rezago (días)").round(3)

**Conclusión:** el 0.48 se reproduce a 1 día, pero la autocorrelación no decae (0.45 a 14 días). Ese patrón corresponde a un cambio de nivel lento, no a un AR(1).

## 2. Estacionalidad

Se ajusta una Normal por mes del año (la misma que usa el modelo) y se revisa si el patrón se repite entre años.

In [ ]:
normales = distribuciones.ajustar_distribuciones_oferta(oferta)
oferta["media_mes"] = oferta["mes"].map(lambda m: normales[m].media)
oferta["desv_mes"] = oferta["mes"].map(lambda m: normales[m].desviacion)

tabla_mes = oferta.pivot_table(index="mes", columns="anio", values="oferta", aggfunc="mean")
tabla_mes.columns = [f"media {a}" for a in tabla_mes.columns]
tabla_mes.insert(0, "media", [normales[m].media for m in tabla_mes.index])
tabla_mes.insert(1, "desviación", [normales[m].desviacion for m in tabla_mes.index])
tabla_mes.index = MESES

print("Media por año:", oferta.groupby("anio")["oferta"].mean().round(1).to_dict())
tabla_mes.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(oferta["fecha"], oferta["oferta"], lw=0.6, color="0.65", label="oferta diaria")
ax.plot(oferta["fecha"], oferta["media_mes"], lw=2, color="C3", label="media del mes")
ax.set_ylabel("unidades")
ax.set_title("Oferta diaria 2022–2024 y media de cada mes")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**Conclusión:** la estacionalidad mensual es fuerte y estable. La media va de ~640 a ~885 según el mes, contra una desviación de ~90 dentro de cada mes; el patrón se repite los tres años y no hay tendencia entre años.

## 3. Autocorrelación sin estacionalidad

Se estandariza la oferta por mes, $z_t = (\text{oferta}_t - \mu_{mes}) / \sigma_{mes}$, y sobre esa serie se calcula la autocorrelación, se estima ρ por regresión de $z_t$ sobre $z_{t-1}$ y se aplica el test de Ljung-Box (H0: no hay autocorrelación).

In [ ]:
oferta["z"] = (oferta["oferta"] - oferta["media_mes"]) / oferta["desv_mes"]
z = oferta["z"].to_numpy()
acf_z = acf(z, nlags=30, fft=False)

print(f"Banda de confianza al 95% para autocorrelación nula: ±{1.96 / np.sqrt(len(z)):.3f}")
pd.DataFrame({
    "rezago (días)": REZAGOS,
    "serie cruda": [acf_cruda[k] for k in REZAGOS],
    "estandarizada por mes": [acf_z[k] for k in REZAGOS],
}).set_index("rezago (días)").round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
plot_acf(oferta["oferta"], lags=30, zero=False, ax=ejes[0], title="Serie cruda")
plot_acf(z, lags=30, zero=False, ax=ejes[1], title="Estandarizada por mes")
for eje in ejes:
    eje.set_xlabel("rezago (días)")
ejes[0].set_ylabel("autocorrelación")
ejes[0].set_ylim(-0.2, 0.7)
plt.tight_layout()
plt.show()

In [ ]:
reg = stats.linregress(z[:-1], z[1:])
ic = 1.96 * reg.stderr
print(f"ρ estimado = {reg.slope:.3f} (IC 95%: {reg.slope - ic:.3f} a {reg.slope + ic:.3f}; valor-p = {reg.pvalue:.2f})")

ljung = acorr_ljungbox(z, lags=[1, 7, 14, 30])
ljung.index.name = "hasta el rezago"
ljung.columns = ["estadístico", "valor-p"]
ljung.round(3)

**Conclusión:** sin la estacionalidad no hay autocorrelación. Todos los rezagos caen dentro de la banda de confianza, el intervalo de ρ contiene al 0 y excluye al 0.48, y Ljung-Box no rechaza la independencia hasta 30 días.

## 4. Robustez

Se repite la estimación (a) estandarizando por mes y año, (b) usando sólo pares de días consecutivos del mismo mes y (c) separando por año. Además, (d) se testea si hay efecto del día de la semana.

In [ ]:
grupo = oferta.groupby(["anio", "mes"])["oferta"]
z_mes_anio = (oferta["oferta"] - grupo.transform("mean")) / grupo.transform(lambda v: v.std(ddof=0))
acf_z2 = acf(z_mes_anio, nlags=30, fft=False)

mismo_mes = oferta["mes"].to_numpy()[1:] == oferta["mes"].to_numpy()[:-1]
rho_mismo_mes = np.corrcoef(z[:-1][mismo_mes], z[1:][mismo_mes])[0, 1]

rho_por_anio = oferta.groupby("anio")["z"].apply(lambda v: v.autocorr(lag=1))

print("(a) Por mes y año:", {k: round(float(acf_z2[k]), 3) for k in REZAGOS})
print(f"(b) Pares dentro del mismo mes: ρ = {rho_mismo_mes:.3f}")
print("(c) ρ a 1 día por año:", rho_por_anio.round(3).to_dict(), f"(banda al 95%: ±{1.96 / np.sqrt(365):.2f})")

In [ ]:
por_dia = oferta.groupby("dia_semana")["z"].agg(["mean", "std", "size"])
por_dia.index = DIAS
por_dia.columns = ["z promedio", "desviación", "días"]
grupos = [g.to_numpy() for _, g in oferta.groupby("dia_semana")["z"]]

print(f"(d) ANOVA: valor-p = {stats.f_oneway(*grupos).pvalue:.2f} | Kruskal-Wallis: valor-p = {stats.kruskal(*grupos).pvalue:.2f}")
por_dia.round(3)

**Conclusión:** ρ ≈ 0 en todas las variantes, y no hay evidencia de un efecto del día de la semana (valores-p de 0.40 y 0.60).

## 5. Comprobación por simulación

Se simulan 500 series de 3 años con el modelo de oferta (Normal del mes + AR(1) sobre $z$), con ρ = 0 y con ρ = 0.48, y se comparan con la historia en la autocorrelación cruda y en la desviación de la oferta semanal (suma de 7 días respecto a su valor esperado).

In [ ]:
def simular_series(rho, n_series, rng):
    n = len(oferta)
    ruido = rng.standard_normal((n_series, n))
    z_sim = np.empty_like(ruido)
    z_sim[:, 0] = ruido[:, 0]
    for t in range(1, n):
        z_sim[:, t] = rho * z_sim[:, t - 1] + np.sqrt(1 - rho**2) * ruido[:, t]
    media, desv = oferta["media_mes"].to_numpy(), oferta["desv_mes"].to_numpy()
    return np.maximum(np.rint(media + desv * z_sim), cfg.OFERTA_MINIMA)


def autocorrelacion(x, k):
    x = x - x.mean()
    return (x[:-k] * x[k:]).sum() / (x * x).sum()


def desviacion_semanal(x):
    exceso = x - oferta["media_mes"].to_numpy()
    semanas = len(exceso) // 7
    return exceso[: semanas * 7].reshape(semanas, 7).sum(axis=1).std()


def resumen(x):
    return {
        "autocorr. cruda, 1 día": autocorrelacion(x, 1),
        "autocorr. cruda, 7 días": autocorrelacion(x, 7),
        "autocorr. cruda, 14 días": autocorrelacion(x, 14),
        "desviación oferta semanal": desviacion_semanal(x),
    }


rng = np.random.default_rng(2024)
filas = {"historia": resumen(oferta["oferta"].to_numpy(dtype=float))}
for rho in (0.0, 0.48):
    sims = pd.DataFrame([resumen(s) for s in simular_series(rho, 500, rng)])
    filas[f"ρ = {rho:g}, promedio"] = sims.mean()
    filas[f"ρ = {rho:g}, percentil 2.5"] = sims.quantile(0.025)
    filas[f"ρ = {rho:g}, percentil 97.5"] = sims.quantile(0.975)

comparacion = pd.DataFrame(filas).T
comparacion.round(3)

**Conclusión:** con ρ = 0 el modelo reproduce por sí solo la autocorrelación cruda de ~0.48, y la historia cae dentro del rango simulado en las cuatro medidas. Con ρ = 0.48 la historia queda fuera del rango en dos de ellas: la autocorrelación a 1 día sube a ~0.72 y la oferta semanal resulta cerca de un 40% más variable que la histórica.

## 6. Conclusión general

- La oferta tiene estacionalidad mensual fuerte y estable, sin tendencia entre años ni efecto del día de la semana. La Normal por mes del año la captura.
- Quitada la estacionalidad, no hay autocorrelación: ρ = 0.013 (IC 95%: −0.046 a 0.072).
- El 0.48 del Informe 1 mide la estacionalidad, no la dependencia entre días. Usarlo como `RHO_OFERTA` cuenta la misma persistencia dos veces.

**Recomendación:** usar `RHO_OFERTA = 0` (oferta Normal del mes, independiente día a día) y mantener `DIAS_RACHA = 1`, ya que no hay dependencia a ningún rezago. La normalidad de la oferta dentro de cada mes no se valida aquí.